<a id="top"></a>

# Lab L1104: build a shallow agent with `create_agent`

```yaml
title: "Lab L1104: build a shallow agent with create_agent"
keywords: create_agent, langchain, langgraph, FakeModel, shallow agent, tools, system prompt, messages, tool_calls, ToolMessage, behavioral equivalence, config surface, offline, lab
estimated duration: 40
```

> **Lesson:** L11. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L11/objectives.md). Solutions: `L1104_lab_solutions.ipynb`. **Runs offline — no API key needed.** You build the agent against a *scripted* `FakeModel`, so every run is deterministic (the same offline approach as the [L1103 demo](L1103_lecture.ipynb)).
>
> **After this lab you can:** build a shallow agent in one `create_agent` call from the shared L10 tools, run it and read the returned message history, pull the final answer off it, and exercise the config surface by swapping the **system prompt** and the **tool set**.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Build the agent in one line](#2-problem-1--build-the-agent-in-one-line)
- [3. Problem 2 — Run it and read the tool sequence](#3-problem-2--run-it-and-read-the-tool-sequence)
- [4. Problem 3 — Pull the final answer off the messages](#4-problem-3--pull-the-final-answer-off-the-messages)
- [5. Problem 4 — Swap the system prompt (config surface)](#5-problem-4--swap-the-system-prompt-config-surface)
- [6. Problem 5 — Swap the tool set (config surface)](#6-problem-5--swap-the-tool-set-config-surface)
- [7. Problem 6 — What did create_agent write for you? (written)](#7-problem-6--what-did-create_agent-write-for-you-written)
- [8. Self-check](#8-self-check)

## 1. Setup

All given: the **same** `calculator`, `lookup`, `flaky_fetch` tools from L10 (imported from `common/tools.py`), the scripted **`FakeModel`** (so this runs offline and deterministically), and a helper `describe(msg)` for printing a message readably. You write **only** the `create_agent` calls and the message-reading code — never a loop.

`create_agent` takes a model, a list of tool callables, and a `system_prompt`, and returns a runnable shallow agent. Invoke it with `agent.invoke({"messages": [...]})`; it returns a dict whose `"messages"` key is the full conversation.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import calculator, flaky_fetch, lookup


def describe(msg: object) -> str:
    """One readable line per message: its type, its text, and any tool calls."""
    kind = type(msg).__name__
    if isinstance(msg, AIMessage) and msg.tool_calls:
        calls = ", ".join(f"{c['name']}({c['args']})" for c in msg.tool_calls)
        return f"{kind:13} -> tool call: {calls}"
    if isinstance(msg, ToolMessage):
        return f"{kind:13} <- result [{msg.status}]: {msg.content!r}"
    text = msg.content if isinstance(msg.content, str) else str(msg.content)
    return f"{kind:13}    {text!r}"


# A scripted model that mimics the L10 chaining run: calculator, then lookup, then answer.
chaining_script = [
    tool_reply(tool_call("c1", "calculator", {"expression": "17**2 - 1"})),
    tool_reply(tool_call("c2", "lookup", {"city": "Lagos"})),
    text_reply("The city is Lagos and its population is 15,000,000."),
]
print("setup ready — tools:", [t.__name__ for t in (calculator, lookup, flaky_fetch)])

[↑ Back to top](#top)

## 2. Problem 1 — Build the agent in one line

Build a shallow agent with **one** `create_agent` call. Pass three things: the model (`FakeModel(chaining_script)`), the tool list (`[calculator, lookup, flaky_fetch]`), and a `system_prompt` telling the assistant to use the tools and not answer from memory. Assign it to `agent`. That's the whole agent — no loop, no routing, no bookkeeping written by you.

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise assistant. Use the provided tools to answer; do not answer from memory."
)

# TODO: build the agent in ONE create_agent call. Pass:
#   1. the model:  FakeModel(chaining_script)
#   2. the tools:  [calculator, lookup, flaky_fetch]
#   3. system_prompt=SYSTEM_PROMPT
# Assign the result to `agent`.
agent = ...  # your code here
print("built:", type(agent).__name__)  # a compiled LangGraph graph

[↑ Back to top](#top)

## 3. Problem 2 — Run it and read the tool sequence

Invoke the agent on the **L10 chaining task** — *the population of the city whose name is the answer to `17**2 - 1`* — by passing a `messages` list with one `HumanMessage`. Save the result to `result`, print every message with `describe`, then build `tool_sequence`: the list of tool **names** the model called, in order. Confirm it is `['calculator', 'lookup']` — the same sequence L10's hand-rolled loop produced.

In [ ]:
chaining_task = (
    "What is the population of the city whose name is the answer to 17**2 - 1? "
    "Use the calculator first, then look the city up."
)

# TODO: invoke the agent on chaining_task (pass {'messages': [HumanMessage(content=...)]}).
#       Save it to `result`.
result = ...  # your code here

# TODO: print every message in result['messages'] with describe(msg).

# TODO: build tool_sequence = the list of tool NAMES called, in order, across all
#       AIMessages (each AIMessage.tool_calls is a list; read call['name']).
tool_sequence = ...  # your code here
print("\ntool sequence:", tool_sequence)  # expect ['calculator', 'lookup']

[↑ Back to top](#top)

## 4. Problem 3 — Pull the final answer off the messages

The agent's final answer is the **content of the last message** in `result["messages"]` (a plain-text `AIMessage` with no tool calls — that's what natural termination looks like). Assign it to `final_answer` and print it.

In [ ]:
# TODO: final_answer = the content of the LAST message in result['messages'].
final_answer = ...  # your code here
print("final answer:", final_answer)

[↑ Back to top](#top)

## 5. Problem 4 — Swap the system prompt (config surface)

The **system prompt** is one of the three knobs on a shallow agent (model, tools, prompt). Build a *second* agent — same model script, same tools — but with a different `system_prompt` that asks the assistant to **answer in exactly one short sentence**. Run it on the same task and print the messages.

This is *the same agent with a different instruction* — the point of the config surface. (With the `FakeModel` the scripted reply text is fixed, so the visible change is the prompt you passed, not the model's wording; against a **live** model the answer's shape would actually change. Either way, you reconfigured behavior without touching the loop.)

In [ ]:
TERSE_PROMPT = (
    "You are a terse assistant. Use the tools, then answer in exactly one short sentence."
)

# TODO: build `terse_agent` with create_agent — SAME model (FakeModel(chaining_script)) and
#       SAME tools, but system_prompt=TERSE_PROMPT. Then invoke it on chaining_task, save to
#       `terse_result`, and print each message with describe.
terse_agent = ...  # your code here
terse_result = ...  # your code here
# your printing loop here

[↑ Back to top](#top)

## 6. Problem 5 — Swap the tool set (config surface)

The **tool list** is another knob. Build an agent given **only** `[calculator]` — no `lookup`, no `flaky_fetch`. Script a `FakeModel` that calls `calculator` once and then answers, run it on a pure-arithmetic task, and confirm the run works with the reduced tool set. The lesson: the tools you pass *are* the agent's whole capability surface; passing fewer is a real configuration choice.

In [ ]:
calc_script = [
    tool_reply(tool_call("a1", "calculator", {"expression": "17**2 - 1"})),
    text_reply("17 squared minus 1 is 288."),
]
# TODO: build `calc_only_agent` with create_agent — model FakeModel(calc_script), tools
#       [calculator] ONLY, system_prompt=SYSTEM_PROMPT. Invoke it on a pure-arithmetic task
#       (e.g. 'What is 17**2 - 1? Use the calculator.'), save to `calc_result`, and print
#       each message with describe.
calc_only_agent = ...  # your code here
calc_result = ...  # your code here
# your printing loop here

[↑ Back to top](#top)

## 7. Problem 6 — What did `create_agent` write for you? (written)

In L10 you wrote the loop by hand. In a sentence or two each, name **two** pieces of that hand-rolled loop that `create_agent` supplied for you here — and for one of them, say what would break if it were missing. (Hint: the `while` driver, the `if reply.tool_calls` routing, the `messages.append(...)` bookkeeping, the `max_steps` / recursion cap, the tool dispatch.)

_Write your answer by editing the cell below (double-click)._

_Your answer here._

[↑ Back to top](#top)

## 8. Self-check

Compare against `L1104_lab_solutions.ipynb`. Done when: `agent` is built in one `create_agent` call; the chaining run prints the `calculator` → `lookup` sequence and `tool_sequence == ['calculator', 'lookup']`; `final_answer` is the last message's content; the terse-prompt and calculator-only agents both run without error; and you can name two things `create_agent` wrote for you (and what breaks without one).

[↑ Back to top](#top)